In [ ]:
import pandas as pd
import glob
import os
import unicodedata

def find_folder_normalized(base_path, target_folder_name):
    # 대상 폴더명을 정규화(NFKD)
    target_norm = unicodedata.normalize('NFKD', target_folder_name)
    for name in os.listdir(base_path):
        if unicodedata.normalize('NFKD', name) == target_norm:
            return os.path.join(base_path, name)
    return None

base_drive_path = '/content/drive/MyDrive'
target_folder = '행정동별 주민등록 인구'

# 실제 매칭되는 폴더 경로 탐색
real_folder_path = find_folder_normalized(base_drive_path, target_folder)

if not real_folder_path:
    # 폴더를 직접 찾지 못한 경우 드라이브 내의 모든 하위 디렉토리를 탐색
    found = False
    for root, dirs, files in os.walk(base_drive_path):
        for d in dirs:
            if unicodedata.normalize('NFKD', d) == unicodedata.normalize('NFKD', target_folder):
                real_folder_path = os.path.join(root, d)
                found = True
                break
        if found:
            break

if real_folder_path and os.path.exists(real_folder_path):
    csv_files = glob.glob(os.path.join(real_folder_path, '*.csv'))
    if not csv_files:
        print(f"폴더를 찾았으나 내부에 CSV 파일이 없습니다: {real_folder_path}")
    else:
        df_list = []
        for file in csv_files:
            try:
                # 한글 CSV 파일에 주로 사용되는 cp949 인코딩으로 시도
                df_temp = pd.read_csv(file, encoding='cp949')
                df_list.append(df_temp)
            except UnicodeDecodeError:
                # 실패할 경우 utf-8 인코딩으로 재시도
                df_temp = pd.read_csv(file, encoding='utf-8')
                df_list.append(df_temp)

        df = pd.concat(df_list, ignore_index=True)
        print(f"성공적으로 '{target_folder}' 폴더를 찾아 {len(csv_files)}개의 파일을 병합했습니다.")
        display(df.head())
else:
    print(f"드라이브 내에서 '{target_folder}' 폴더를 찾을 수 없습니다. 폴더명을 다시 확인해주세요.")

In [ ]:
import os
import pandas as pd

# 각 파일별 열 이름(기간) 요약 및 데이터 범위 확인
print("=== 파일별 데이터 기간(열) 분석 ===")
for file in csv_files:
    file_name = os.path.basename(file)
    try:
        temp_df = pd.read_csv(file, encoding='cp949', nrows=1)
    except UnicodeDecodeError:
        temp_df = pd.read_csv(file, encoding='utf-8', nrows=1)

    # '인구수' 또는 '총인구수'가 포함된 열들만 필터링하여 기간 추출
    date_cols = [col for col in temp_df.columns if '인구수' in col]
    # 정렬하여 시작월과 종료월 파악
    date_cols_sorted = sorted(date_cols)

    if date_cols_sorted:
        print(f"\n파일: {file_name}")
        print(f"  - 열 개수: {len(temp_df.columns)}개")
        print(f"  - 데이터 기간: {date_cols_sorted[0]} ~ {date_cols_sorted[-1]}")
    else:
        print(f"\n파일: {file_name} (날짜 관련 열을 찾을 수 없음)")

In [ ]:
import os
import pandas as pd

# 파일별 인구 컬럼 구조 파악
print("=== 파일별 인구 컬럼 형태 확인 ===")

only_total_pop_files = []
resident_pop_files = []

for file in csv_files:
    file_name = os.path.basename(file)
    try:
        temp_df = pd.read_csv(file, encoding='cp949', nrows=1)
    except UnicodeDecodeError:
        temp_df = pd.read_csv(file, encoding='utf-8', nrows=1)

    # '총인구수'와 '거주자 인구수' 컬럼 존재 여부 체크
    has_total = any('총인구수' in col for col in temp_df.columns)
    has_resident = any('거주자 인구수' in col for col in temp_df.columns)

    if has_total and not has_resident:
        only_total_pop_files.append(file_name)
    elif has_resident:
        resident_pop_files.append(file_name)

print("\n1. [총인구수] 컬럼만 존재하는 파일 목록:")
for f in only_total_pop_files:
    print(f"  - {f}")

print("\n2. [거주자 인구수] 컬럼이 포함되어 있는 파일 목록:")
for f in resident_pop_files:
    print(f"  - {f}")